imports

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
import pandas as pd
import numpy as np
import plotly.express as px
from gencost.crosswalk import Crosswalk
from gencost.waterfall import DataBySubplant

warnings.simplefilter(action="once")

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
import logging
import shutil
import warnings
from datetime import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pyarrow
import pyarrow.dataset as ds
from etoolbox.utils.pudl_helpers import (
    month_year_to_date,
    simplify_columns,
    sum_and_weighted_average_agg,
)
from etoolbox.utils.remote_zip import RemoteIOError, RemoteZip
from platformdirs import user_cache_path, user_documents_path
from tqdm.auto import tqdm
from tqdm.contrib.logging import logging_redirect_tqdm

from gencost.constants import FOSSIL_PRIME_MOVER_MAP, FUEL_GROUP_MAP
from gencost.crosswalk import Crosswalk
from gencost.package_data import PACKAGE_PATH

#pat_path = Path(__file__).parent
CACHE_PATH = user_cache_path("gencost", "rmi")
logger = logging.getLogger(__name__)


data source objects

In [43]:
xwalk = Crosswalk()
# self to be able to copy / paste from waterfall.py for dev ease
self = DataBySubplant(xwalk)

In [ ]:
[
                [GET_860_GEN_COLS]
            ]

In [64]:
big = self.get_860_by_x(generator_level=True)

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:285: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(


In [65]:
big

,plant_id_eia,generator_id,report_date,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,...,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_from_report_year,avg_age_from_report_year,current_avg_age,age_of_observation,age_relative_to_avg,pollution_control_costs_per_kw
0,2,1,2001-01-01,45.0,False,False,False,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,37.505818,48.004853,58.420260,20.914442,10.415407,2.92732
1,3,1,2001-01-01,153.1,False,False,False,<NA>,<NA>,<NA>,...,<NA>,True,<NA>,<NA>,46.915811,57.414847,67.830253,20.914442,10.415407,2.92732
2,3,2,2001-01-01,153.1,False,False,False,<NA>,<NA>,<NA>,...,<NA>,True,<NA>,<NA>,46.505133,57.004169,67.419576,20.914442,10.415407,2.92732
3,3,3,2001-01-01,272.0,False,False,False,<NA>,<NA>,<NA>,...,<NA>,True,<NA>,<NA>,41.505818,48.004889,62.420260,20.914442,14.415371,NaN
4,3,4,2001-01-01,403.7,False,False,False,<NA>,<NA>,<NA>,...,<NA>,True,<NA>,<NA>,31.085558,41.584593,52.000000,20.914442,10.415407,2.92732
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427124,65774,18031,2022-01-01,2.0,False,False,False,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,1.084189,0.584531,0.999316,-0.084873,0.414784,NaN
427125,65775,5324,2022-01-01,2.0,False,False,False,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,1.084189,0.584531,0.999316,-0.084873,0.414784,NaN
427126,65776,5939,2022-01-01,2.0,False,False,False,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,1.251198,0.751540,1.166324,-0.084873,0.414784,NaN
427127,65777,5779,2022-01-01,2.0,False,False,False,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,1.333333,0.833676,1.248460,-0.084873,0.414784,NaN


In [55]:
big.columns.tolist()

['report_date',
 'plant_id_eia',
 'plant_id_pudl',
 'plant_name_eia',
 'utility_id_eia',
 'utility_id_pudl',
 'utility_name_eia',
 'generator_id',
 'associated_combined_heat_power',
 'bga_source',
 'bypass_heat_recovery',
 'capacity_mw',
 'carbon_capture',
 'city',
 'cofire_fuels',
 'county',
 'current_planned_generator_operating_date',
 'data_maturity',
 'deliver_power_transgrid',
 'distributed_generation',
 'duct_burners',
 'energy_source_1_transport_1',
 'energy_source_1_transport_2',
 'energy_source_1_transport_3',
 'energy_source_2_transport_1',
 'energy_source_2_transport_2',
 'energy_source_2_transport_3',
 'energy_source_code_1',
 'energy_source_code_2',
 'energy_source_code_3',
 'energy_source_code_4',
 'energy_source_code_5',
 'energy_source_code_6',
 'energy_storage_capacity_mwh',
 'ferc_qualifying_facility',
 'fluidized_bed_tech',
 'fuel_type_code_pudl',
 'fuel_type_count',
 'generator_operating_date',
 'generator_retirement_date',
 'latitude',
 'longitude',
 'minimum_load_

In [9]:
xwalk = {"pf_subplant_id": self.safe_xwalk, "subplant_id": self.xwalk}[
            'pf_subplant_id'
        ]
xwalk

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:285: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk


,plant_id_eia,generator_id,emissions_unit_id_epa,subplant_id,pf_subplant_id,prime_mover,fuel_group,unit_id_pudl,capacity_xwalk,generator_operating_date,plant_id_epa,ppf,single_prime
0,1,WT2,WT2,0,0,WT,renew,<NA>,0.5,2011-10-01,1,1_WT_renew,True
1,1,WT1,WT1,1,0,WT,renew,<NA>,0.5,2011-10-01,1,1_WT_renew,True
2,1,5,5,2,1,IC,petroleum,<NA>,0.7,2000-12-01,1,1_IC_petroleum,True
3,1,3,3,3,1,IC,petroleum,<NA>,0.5,2010-12-01,1,1_IC_petroleum,True
4,1,2,2,4,1,IC,petroleum,<NA>,0.9,2000-12-01,1,1_IC_petroleum,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43083,65815,MPRES,MPRES,0,0,PV,renew,<NA>,1.0,2021-12-01,65815,65815_PV_renew,True
43084,65817,CLEA1,CLEA1,0,0,PV,renew,<NA>,3.6,2020-09-01,65817,65817_PV_renew,True
43085,65824,VSPRC,VSPRC,0,0,PV,renew,<NA>,1.1,2020-04-01,65824,65824_PV_renew,True
43086,65836,TOYAH,TOYAH,0,0,BA,charge,<NA>,10.4,2021-10-01,65836,65836_BA_charge,True


In [10]:
xwalk = {"pf_subplant_id": self.safe_xwalk, "subplant_id": self.xwalk}[
            'subplant_id'
        ]
xwalk

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:285: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk


,plant_id_eia,generator_id,emissions_unit_id_epa,subplant_id,pf_subplant_id,prime_mover,fuel_group,unit_id_pudl,capacity_xwalk,generator_operating_date,plant_id_epa,ppf,single_prime
0,1,WT2,WT2,0,0,WT,renew,<NA>,0.5,2011-10-01,1,1_WT_renew,True
1,1,WT1,WT1,1,0,WT,renew,<NA>,0.5,2011-10-01,1,1_WT_renew,True
2,1,5,5,2,1,IC,petroleum,<NA>,0.7,2000-12-01,1,1_IC_petroleum,True
3,1,3,3,3,1,IC,petroleum,<NA>,0.5,2010-12-01,1,1_IC_petroleum,True
4,1,2,2,4,1,IC,petroleum,<NA>,0.9,2000-12-01,1,1_IC_petroleum,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43083,65815,MPRES,MPRES,0,0,PV,renew,<NA>,1.0,2021-12-01,65815,65815_PV_renew,True
43084,65817,CLEA1,CLEA1,0,0,PV,renew,<NA>,3.6,2020-09-01,65817,65817_PV_renew,True
43085,65824,VSPRC,VSPRC,0,0,PV,renew,<NA>,1.1,2020-04-01,65824,65824_PV_renew,True
43086,65836,TOYAH,TOYAH,0,0,BA,charge,<NA>,10.4,2021-10-01,65836,65836_BA_charge,True


what do we need in get exa by generator?

In [5]:
xwalk

,plant_id_eia,generator_id,emissions_unit_id_epa,subplant_id,pf_subplant_id,prime_mover,fuel_group,unit_id_pudl,capacity_xwalk,generator_operating_date,plant_id_epa,ppf,single_prime
0,1,WT2,WT2,0,0,WT,renew,<NA>,0.5,2011-10-01,1,1_WT_renew,True
1,1,WT1,WT1,1,0,WT,renew,<NA>,0.5,2011-10-01,1,1_WT_renew,True
2,1,5,5,2,1,IC,petroleum,<NA>,0.7,2000-12-01,1,1_IC_petroleum,True
3,1,3,3,3,1,IC,petroleum,<NA>,0.5,2010-12-01,1,1_IC_petroleum,True
4,1,2,2,4,1,IC,petroleum,<NA>,0.9,2000-12-01,1,1_IC_petroleum,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43083,65815,MPRES,MPRES,0,0,PV,renew,<NA>,1.0,2021-12-01,65815,65815_PV_renew,True
43084,65817,CLEA1,CLEA1,0,0,PV,renew,<NA>,3.6,2020-09-01,65817,65817_PV_renew,True
43085,65824,VSPRC,VSPRC,0,0,PV,renew,<NA>,1.1,2020-04-01,65824,65824_PV_renew,True
43086,65836,TOYAH,TOYAH,0,0,BA,charge,<NA>,10.4,2021-10-01,65836,65836_BA_charge,True


In [3]:
self.get_860_by_x("generator_id")

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:285: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk


KeyError: 'generator_id'

In [40]:
def get_860_by_x(self,subplant_id_col="pf_subplant_id",merge_only=False,age_year=2021,generator_level=False):
    """
    input: unaggregated 860, 923, CAMD data

    output: 860, 923, CAMD data at desired sub-plant level

    """

    xwalk = {"pf_subplant_id": self.safe_xwalk, "subplant_id": self.xwalk}[
            subplant_id_col
        ]

    coi = (
            pd.read_parquet(PACKAGE_PATH / "unit_level_costs_with_flag.parquet.gzip")
            .pipe(simplify_columns)
            .pipe(month_year_to_date)
            .rename(columns={"plant_id": "plant_id_eia"})
        )[["plant_id_eia", "generator_id", "pollution_control_costs_per_kw"]]

    if generator_level == False:
        merged = (
                self.pudl_tabl.gens_eia860()
                .query("operational_status == 'existing'")
                .assign(
                    prime_mover=lambda x: x.prime_mover_code.replace(
                        FOSSIL_PRIME_MOVER_MAP
                    ),
                )
                .copy()
                .merge(
                    xwalk[
                        ["plant_id_eia", "generator_id", subplant_id_col]
                    ].drop_duplicates(),
                    on=["plant_id_eia", "generator_id"],
                    how="outer",
                    validate="m:1",
                    indicator=True,
                )
                .merge(
                    coi[["plant_id_eia", "generator_id", "pollution_control_costs_per_kw"]],
                    on=["plant_id_eia", "generator_id"],
                    how="left",
                    validate="m:1",
                ))
        test = (
            merged.query(
                "_merge != 'both' & prime_mover_code in @FOSSIL_PRIME_MOVER_MAP"
            )
            .replace(
                {"_merge": {"left_only": "in_data_only", "right_only": "in_xwalk_only"}}
            )
            .groupby(["_merge", "prime_mover_code"], dropna=False)
            .plant_id_eia.nunique()
            .to_frame()
            .query("plant_id_eia > 0")
        )

        logger.warning(
            "860 %s: Unique plants that did not have matches in both "
            "860 and the xwalk so will be dropped:\\n %s \\n",
            {"pf_subplant_id": "prime", "subplant_id": "subplant"}[subplant_id_col],
            test.squeeze().to_dict(),
        )
        if merge_only:
            return merged
        wtavg_dict = {
            "associated_combined_heat_power": "capacity_mw",
            "duct_burners": "capacity_mw",
            "bypass_heat_recovery": "capacity_mw",
            "solid_fuel_gasification": "capacity_mw",
            "carbon_capture": "capacity_mw",
            "fluidized_bed_tech": "capacity_mw",
            "pulverized_coal_tech": "capacity_mw",
            "stoker_tech": "capacity_mw",
            "other_combustion_tech": "capacity_mw",
            "subcritical_tech": "capacity_mw",
            "supercritical_tech": "capacity_mw",
            "ultrasupercritical_tech": "capacity_mw",
            "age_from_report_year": "capacity_mw",
            "avg_age_from_report_year": "capacity_mw",
            "current_avg_age": "capacity_mw",
            "age_of_observation": "capacity_mw",
            "age_relative_to_avg": "capacity_mw",
            "pollution_control_costs_per_kw": "capacity_mw",
        }

        if age_year is not None:
            age_year_str = dt.strptime(f"12-1-{age_year}", "%m-%d-%Y")
        else:
            age_year_str = dt.utcnow()
        return (
            merged.query("_merge == 'both'")
            .assign(
                age_from_report_year=lambda x: (
                    x["report_date"] - x["generator_operating_date"]
                ).dt.days
                / 365.25,
                avg_age_from_report_year=lambda x: x.groupby(
                    ["plant_id_eia", subplant_id_col]
                )["age_from_report_year"].transform("mean"),
                current_age=lambda x: (
                    age_year_str - x["generator_operating_date"]
                ).dt.days
                / 365.25,
                current_avg_age=lambda x: x.groupby(["plant_id_eia", subplant_id_col])[
                    "current_age"
                ].transform("mean"),
                age_of_observation=lambda x: (age_year_str - x["report_date"]).dt.days
                / 365.25,
                age_relative_to_avg=lambda x: x["current_age"]
                - x["avg_age_from_report_year"],
            )
            .astype({k: float for k in wtavg_dict})
            .fillna({k: 0.0 for k in wtavg_dict})
            .drop(columns=["_merge"])
            .pipe(
                sum_and_weighted_average_agg,
                by=[
                    "plant_id_eia",
                    subplant_id_col,
                    pd.Grouper(key="report_date", freq="YS"),
                ],
                sum_cols=["capacity_mw"],
                wtavg_dict=wtavg_dict,
            )
            .astype({"plant_id_eia": "Int64", subplant_id_col: "Int64"})
        )
    else:
        merged = (
                self.pudl_tabl.gens_eia860()
                .query("operational_status == 'existing'")
                .assign(
                    prime_mover=lambda x: x.prime_mover_code.replace(
                        FOSSIL_PRIME_MOVER_MAP
                    ),
                )
                .copy()
                .merge(
                    coi[["plant_id_eia", "generator_id", "pollution_control_costs_per_kw"]],
                    on=["plant_id_eia", "generator_id"],
                    how="left",
                    validate="m:1",
                    indicator=True
                ))

        test = (
            merged.query(
                "_merge != 'both' & prime_mover_code in @FOSSIL_PRIME_MOVER_MAP"
            )
            .replace(
                {"_merge": {"left_only": "in_data_only", "right_only": "in_xwalk_only"}}
            )
            .groupby(["_merge", "prime_mover_code"], dropna=False)
            .plant_id_eia.nunique()
            .to_frame()
            .query("plant_id_eia > 0")
        )

        if merge_only:
            return merged
     
        if age_year is not None:
            age_year_str = dt.strptime(f"12-1-{age_year}", "%m-%d-%Y")
        else:
            age_year_str = dt.utcnow()
        return (
            merged.query("_merge == 'both'")
            .assign(
                age_from_report_year=lambda x: (
                    x["report_date"] - x["generator_operating_date"]
                ).dt.days
                / 365.25,
                avg_age_from_report_year=lambda x: x.groupby(
                    ["plant_id_eia", "generator_id"]
                )["age_from_report_year"].transform("mean"),
                current_age=lambda x: (
                    age_year_str - x["generator_operating_date"]
                ).dt.days
                / 365.25,
                current_avg_age=lambda x: x.groupby(["plant_id_eia", "generator_id"])[
                    "current_age"
                ].transform("mean"),
                age_of_observation=lambda x: (age_year_str - x["report_date"]).dt.days
                / 365.25,
                age_relative_to_avg=lambda x: x["current_age"]
                - x["avg_age_from_report_year"],
            )
            .drop(columns=["_merge"])
            .astype({"plant_id_eia": "Int64","generator_id":"Int64"})
        )



    #return test

SyntaxError: expression expected after dictionary key and ':' (161637557.py, line 190)

In [39]:
get_860_by_x(self,"pf_subplant_id",generator_level=True)

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:285: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(


,report_date,plant_id_eia,plant_id_pudl,plant_name_eia,utility_id_eia,utility_id_pudl,utility_name_eia,generator_id,associated_combined_heat_power,bga_source,...,winter_estimated_capability_mw,zip_code,prime_mover,pollution_control_costs_per_kw,age_from_report_year,avg_age_from_report_year,current_age,current_avg_age,age_of_observation,age_relative_to_avg
0,2001-01-01,2,848,Bankhead Dam,195,18,Alabama Power Co,1,False,<NA>,...,NaN,35476,HY,2.92732,37.505818,48.004853,58.420260,58.420260,20.914442,10.415407
1,2001-01-01,3,32,Barry,195,18,Alabama Power Co,1,False,<NA>,...,NaN,36512,ST,2.92732,46.915811,57.414847,67.830253,67.830253,20.914442,10.415407
2,2001-01-01,3,32,Barry,195,18,Alabama Power Co,2,False,<NA>,...,NaN,36512,ST,2.92732,46.505133,57.004169,67.419576,67.419576,20.914442,10.415407
3,2001-01-01,3,32,Barry,195,18,Alabama Power Co,3,False,<NA>,...,NaN,36512,ST,NaN,41.505818,48.004889,62.420260,62.420260,20.914442,14.415371
4,2001-01-01,3,32,Barry,195,18,Alabama Power Co,4,False,<NA>,...,NaN,36512,ST,2.92732,31.085558,41.584593,52.000000,52.000000,20.914442,10.415407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427124,2022-01-01,65774,17019,Alden Road Harvard Solar 1,65024,14228,"Alden Road Harvard Solar 1, LLC",18031,False,<NA>,...,NaN,60033,PV,NaN,1.084189,0.584531,0.999316,0.999316,-0.084873,0.414784
427125,2022-01-01,65775,17020,Cream Street Solar,65030,14231,"Cream Street Solar, LLC",5324,False,<NA>,...,NaN,12538,PV,NaN,1.084189,0.584531,0.999316,0.999316,-0.084873,0.414784
427126,2022-01-01,65776,17021,Little Falls Solar,65028,14229,Little Falls Solar,5939,False,<NA>,...,NaN,13365,PV,NaN,1.251198,0.751540,1.166324,1.166324,-0.084873,0.414784
427127,2022-01-01,65777,17022,Little Falls Solar 1,65029,14230,"Little Falls Solar 1, LLC",5779,False,<NA>,...,NaN,13365,PV,NaN,1.333333,0.833676,1.248460,1.248460,-0.084873,0.414784


In [41]:
df = get_860_by_x(self,"pf_subplant_id",generator_level=False)

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:285: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
860 prime: Unique plants that did not have matches in both 860 and the xwalk so will be dropped:\n {('in_data_only', 'CA'): 42, ('in_data_only', 'CS'): 4, ('in_data_only', 'CT'): 43, ('in_data_only', 'GT'): 55, ('in_data_only', 'IC'): 100, ('in_data_only', 'ST'): 83} \n


In [42]:
df.columns.to_list()

['plant_id_eia',
 'pf_subplant_id',
 'report_date',
 'capacity_mw',
 'associated_combined_heat_power',
 'duct_burners',
 'bypass_heat_recovery',
 'solid_fuel_gasification',
 'carbon_capture',
 'fluidized_bed_tech',
 'pulverized_coal_tech',
 'stoker_tech',
 'other_combustion_tech',
 'subcritical_tech',
 'supercritical_tech',
 'ultrasupercritical_tech',
 'age_from_report_year',
 'avg_age_from_report_year',
 'current_avg_age',
 'age_of_observation',
 'age_relative_to_avg',
 'pollution_control_costs_per_kw']

In [ ]:

    logger.warning(
        "860 %s: Unique plants that did not have matches in both "
        "860 and the xwalk so will be dropped:\\n %s \\n",
        {"pf_subplant_id": "prime", "subplant_id": "subplant"}[subplant_id_col],
        test.squeeze().to_dict(),
    )
    if merge_only:
        return merged
    wtavg_dict = {
        "associated_combined_heat_power": "capacity_mw",
        "duct_burners": "capacity_mw",
        "bypass_heat_recovery": "capacity_mw",
        "solid_fuel_gasification": "capacity_mw",
        "carbon_capture": "capacity_mw",
        "fluidized_bed_tech": "capacity_mw",
        "pulverized_coal_tech": "capacity_mw",
        "stoker_tech": "capacity_mw",
        "other_combustion_tech": "capacity_mw",
        "subcritical_tech": "capacity_mw",
        "supercritical_tech": "capacity_mw",
        "ultrasupercritical_tech": "capacity_mw",
        "age_from_report_year": "capacity_mw",
        "avg_age_from_report_year": "capacity_mw",
        "current_avg_age": "capacity_mw",
        "age_of_observation": "capacity_mw",
        "age_relative_to_avg": "capacity_mw",
        "pollution_control_costs_per_kw": "capacity_mw",
    }

    if age_year is not None:
        age_year_str = dt.strptime(f"12-1-{age_year}", "%m-%d-%Y")
    else:
        age_year_str = dt.utcnow()
    return (
        merged.query("_merge == 'both'")
        .assign(
            age_from_report_year=lambda x: (
                x["report_date"] - x["generator_operating_date"]
            ).dt.days
            / 365.25,
            avg_age_from_report_year=lambda x: x.groupby(
                ["plant_id_eia", subplant_id_col]
            )["age_from_report_year"].transform("mean"),
            current_age=lambda x: (
                age_year_str - x["generator_operating_date"]
            ).dt.days
            / 365.25,
            current_avg_age=lambda x: x.groupby(["plant_id_eia", subplant_id_col])[
                "current_age"
            ].transform("mean"),
            age_of_observation=lambda x: (age_year_str - x["report_date"]).dt.days
            / 365.25,
            age_relative_to_avg=lambda x: x["current_age"]
            - x["avg_age_from_report_year"],
        )
        .astype({k: float for k in wtavg_dict})
        .fillna({k: 0.0 for k in wtavg_dict})
        .drop(columns=["_merge"])
        .pipe(
            sum_and_weighted_average_agg,
            by=[
                "plant_id_eia",
                subplant_id_col,
                pd.Grouper(key="report_date", freq="YS"),
            ],
            sum_cols=["capacity_mw"],
            wtavg_dict=wtavg_dict,
        )
        .astype({"plant_id_eia": "Int64", subplant_id_col: "Int64"})
    )